<a href="https://colab.research.google.com/github/KARANMAJILA/mindsdb/blob/main/meetingagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


tasks = [
    {
        "task_id": "T101",
        "description": "Prepare the deployment plan",
        "assignee": "Karan Majila",
        "status": "Open",
        "due_date": "2026-11-02",
        "meeting_id": "M01"
    },
    {
        "task_id": "T102",
        "description": "Test the authentication API",
        "assignee": "Karan Majila and team",
        "status": "In Progress",
        "due_date": "2026-11-04",
        "meeting_id": "M01"
    },
    {
        "task_id": "T103",
        "description": "Prepare the UI design",
        "assignee": "Priya Sharma",
        "status": "Open",
        "due_date": "2026-11-05",
        "meeting_id": "M02"
    }
]

meetings = [
    {
        "meeting_id": "M01",
        "summary": "The team discussed deployment and API authentication.",
        "decisions": "Use token-based authentication and test before deployment."
    },
    {
        "meeting_id": "M02",
        "summary": "The team reviewed the user interface.",
        "decisions": "Keep the UI simple and consistent across screens."
    }
]

print("Tasks loaded:", len(tasks))
print("Meetings loaded:", len(meetings))


Tasks loaded: 3
Meetings loaded: 2


In [2]:

# Cell 2: Task lookup tool

def get_action_items(employee_name, status=None):
    results = []

    for task in tasks:
        # Match the employee name anywhere in the assignee field
        if employee_name.casefold() in task["assignee"].casefold():
            if status is None or task["status"].casefold() == status.casefold():
                results.append(task)

    return results


# Test the tool
get_action_items("Karan Majila")


[{'task_id': 'T101',
  'description': 'Prepare the deployment plan',
  'assignee': 'Karan Majila',
  'status': 'Open',
  'due_date': '2026-11-02',
  'meeting_id': 'M01'},
 {'task_id': 'T102',
  'description': 'Test the authentication API',
  'assignee': 'Karan Majila and team',
  'status': 'In Progress',
  'due_date': '2026-11-04',
  'meeting_id': 'M01'}]

In [3]:

#Meeting search tool

def search_meetings(query):
    query_words = query.casefold().split()
    results = []

    for meeting in meetings:
        text = (
            meeting["summary"] + " " +
            meeting["decisions"]
        ).casefold()

        matched_words = sum(word in text for word in query_words)

        if matched_words > 0:
            results.append({
                **meeting,
                "matched_words": matched_words
            })

    return sorted(
        results,
        key=lambda item: item["matched_words"],
        reverse=True
    )


search_meetings("authentication deployment")


[{'meeting_id': 'M01',
  'summary': 'The team discussed deployment and API authentication.',
  'decisions': 'Use token-based authentication and test before deployment.',
  'matched_words': 2}]

In [4]:

# Simple tool router

def run_agent(question):
    q = question.casefold()

    if any(word in q for word in [
        "assigned", "action item", "tasks", "overdue"
    ]):
        employee = None

        # Demo-only entity extraction
        for name in ["Karan Majila", "Priya Sharma"]:
            if name.casefold() in q:
                employee = name
                break

        if employee:
            result = get_action_items(employee)
            return {
                "tool": "get_action_items",
                "result": result
            }

        return {
            "tool": "get_action_items",
            "result": "Please specify an employee name."
        }

    result = search_meetings(question)
    return {
        "tool": "search_meetings",
        "result": result
    }


run_agent("Which tasks are assigned to Karan Majila?")


{'tool': 'get_action_items',
 'result': [{'task_id': 'T101',
   'description': 'Prepare the deployment plan',
   'assignee': 'Karan Majila',
   'status': 'Open',
   'due_date': '2026-11-02',
   'meeting_id': 'M01'},
  {'task_id': 'T102',
   'description': 'Test the authentication API',
   'assignee': 'Karan Majila and team',
   'status': 'In Progress',
   'due_date': '2026-11-04',
   'meeting_id': 'M01'}]}

In [5]:



import time
import json

def traced_agent(question):
    start = time.perf_counter()

    print("TRACE | question:", question)

    result = run_agent(question)

    elapsed_ms = (time.perf_counter() - start) * 1000

    print("TRACE | selected_tool:", result["tool"])
    print("TRACE | elapsed_ms:", round(elapsed_ms, 2))
    print("TRACE | result_count:",
          len(result["result"])
          if isinstance(result["result"], list)
          else "N/A")
    print("TRACE | result:", json.dumps(
        result["result"], indent=2, default=str
    ))

    return result


traced_agent("Which tasks are assigned to Karan Majila?")


TRACE | question: Which tasks are assigned to Karan Majila?
TRACE | selected_tool: get_action_items
TRACE | elapsed_ms: 0.11
TRACE | result_count: 2
TRACE | result: [
  {
    "task_id": "T101",
    "description": "Prepare the deployment plan",
    "assignee": "Karan Majila",
    "status": "Open",
    "due_date": "2026-11-02",
    "meeting_id": "M01"
  },
  {
    "task_id": "T102",
    "description": "Test the authentication API",
    "assignee": "Karan Majila and team",
    "status": "In Progress",
    "due_date": "2026-11-04",
    "meeting_id": "M01"
  }
]


{'tool': 'get_action_items',
 'result': [{'task_id': 'T101',
   'description': 'Prepare the deployment plan',
   'assignee': 'Karan Majila',
   'status': 'Open',
   'due_date': '2026-11-02',
   'meeting_id': 'M01'},
  {'task_id': 'T102',
   'description': 'Test the authentication API',
   'assignee': 'Karan Majila and team',
   'status': 'In Progress',
   'due_date': '2026-11-04',
   'meeting_id': 'M01'}]}

In [6]:
traced_agent("Which tasks are assigned to Karan Majila?")

TRACE | question: Which tasks are assigned to Karan Majila?
TRACE | selected_tool: get_action_items
TRACE | elapsed_ms: 1.36
TRACE | result_count: 2
TRACE | result: [
  {
    "task_id": "T101",
    "description": "Prepare the deployment plan",
    "assignee": "Karan Majila",
    "status": "Open",
    "due_date": "2026-11-02",
    "meeting_id": "M01"
  },
  {
    "task_id": "T102",
    "description": "Test the authentication API",
    "assignee": "Karan Majila and team",
    "status": "In Progress",
    "due_date": "2026-11-04",
    "meeting_id": "M01"
  }
]


{'tool': 'get_action_items',
 'result': [{'task_id': 'T101',
   'description': 'Prepare the deployment plan',
   'assignee': 'Karan Majila',
   'status': 'Open',
   'due_date': '2026-11-02',
   'meeting_id': 'M01'},
  {'task_id': 'T102',
   'description': 'Test the authentication API',
   'assignee': 'Karan Majila and team',
   'status': 'In Progress',
   'due_date': '2026-11-04',
   'meeting_id': 'M01'}]}

In [7]:
traced_agent("Which tasks are assigned to Priya Sharma?")

TRACE | question: Which tasks are assigned to Priya Sharma?
TRACE | selected_tool: get_action_items
TRACE | elapsed_ms: 0.1
TRACE | result_count: 1
TRACE | result: [
  {
    "task_id": "T103",
    "description": "Prepare the UI design",
    "assignee": "Priya Sharma",
    "status": "Open",
    "due_date": "2026-11-05",
    "meeting_id": "M02"
  }
]


{'tool': 'get_action_items',
 'result': [{'task_id': 'T103',
   'description': 'Prepare the UI design',
   'assignee': 'Priya Sharma',
   'status': 'Open',
   'due_date': '2026-11-05',
   'meeting_id': 'M02'}]}

In [8]:

%pip -q install -U openai


In [11]:

from getpass import getpass
from openai import OpenAI

groq_api_key = getpass("Enter your Groq API key: ")

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Groq client initialized!")


Enter your Groq API key: ··········
Groq client initialized!


In [12]:

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain AI tool calling in three simple sentences."
        }
    ]
)

print(response.choices[0].message.content)


AI tool calling lets a large language model decide when it needs help from another program or service.  
It then emits a structured request—specifying the tool name and any needed arguments—to that external function.  
The system runs the tool, returns its result to the model, and the model uses that answer to complete the user’s task.


In [13]:
import json
import time

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_action_items",
            "description": "Find tasks assigned to an employee, optionally filtered by status.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_name": {
                        "type": "string",
                        "description": "Employee's full name"
                    },
                    "status": {
                        "type": ["string", "null"],
                        "description": "Optional status such as Open or In Progress"
                    }
                },
                "required": ["employee_name", "status"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_meetings",
            "description": "Search meeting summaries and decisions by topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Meeting topic or question to search"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    }
]

print("Tools defined:", [t["function"]["name"] for t in tools])

Tools defined: ['get_action_items', 'search_meetings']


In [14]:
def execute_tool(name, arguments):
    if name == "get_action_items":
        return get_action_items(
            employee_name=arguments["employee_name"],
            status=arguments["status"]
        )

    elif name == "search_meetings":
        return search_meetings(arguments["query"])

    else:
        raise ValueError(f"Unknown tool: {name}")

In [21]:
def ask_agent(question):
    messages = [
        {
            "role": "system",
            "content": """
            You are a meeting assistant.
            Use get_action_items for task and assignment questions.
            Use search_meetings for meeting discussions and decisions.
            Never invent task records or meeting decisions.
            Base factual answers on tool results.
            If no matching records are found, say so.
            """
        },
        {"role": "user", "content": question}
    ]

    start = time.perf_counter()

    for _ in range(5):  # Limit tool-call rounds
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        if not message.tool_calls:
            print(
                "TRACE | total_latency_ms:",
                round((time.perf_counter() - start) * 1000, 2)
            )
            return message.content

        messages.append(message)

        for call in message.tool_calls:
            name = call.function.name
            arguments = json.loads(call.function.arguments)

            print("TRACE | tool:", name)
            print("TRACE | arguments:", arguments)

            try:
                result = execute_tool(name, arguments)
                tool_result = json.dumps(result, default=str)
            except Exception as e:
                tool_result = json.dumps({"error": str(e)})

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": tool_result
            })

    return "I couldn't complete the request within the tool-call limit."

In [24]:
# Initialize a global conversation history with the system prompt
conversation_history = [
    {
        "role": "system",
        "content": """
        You are a meeting assistant.
        Use get_action_items for task and assignment questions.
        Use search_meetings for meeting discussions and decisions.
        Never invent task records or meeting decisions.
        Base factual answers on tool results.
        If no matching records are found, say so.
        """
    }
]

def ask_agent(question):
    global conversation_history

    # Append user question to the history
    conversation_history.append({"role": "user", "content": question})

    start = time.perf_counter()

    for _ in range(5):  # Limit tool-call rounds
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=conversation_history,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        if not message.tool_calls:
            print(
                "TRACE | total_latency_ms:",
                round((time.perf_counter() - start) * 1000, 2)
            )
            # Save the model's final response to history before returning
            conversation_history.append({"role": "assistant", "content": message.content})
            return message.content

        conversation_history.append(message)

        for call in message.tool_calls:
            name = call.function.name
            arguments = json.loads(call.function.arguments)

            print("TRACE | tool:", name)
            print("TRACE | arguments:", arguments)

            try:
                result = execute_tool(name, arguments)
                tool_result = json.dumps(result, default=str)
            except Exception as e:
                tool_result = json.dumps({"error": str(e)})

            conversation_history.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": tool_result
            })

    return "I couldn't complete the request within the tool-call limit."

In [16]:
print(ask_agent("Which tasks are assigned to Karan Majila?"))

TRACE | tool: get_action_items
TRACE | arguments: {'employee_name': 'Karan Majila', 'status': None}
TRACE | total_latency_ms: 772.55
**Tasks assigned to Karan Majila**

| Task ID | Description | Status | Due Date | Meeting ID |
|---------|-------------|--------|----------|------------|
| T101 | Prepare the deployment plan | Open | 2026‑11‑02 | M01 |
| T102 | Test the authentication API | In Progress | 2026‑11‑04 | M01 |

*Note: Task T102 is shared with the team, but Karan Majila is listed as an assignee.*


In [17]:
print(ask_agent("What did the team decide about authentication?"))

TRACE | tool: search_meetings
TRACE | arguments: {'query': 'authentication'}
TRACE | total_latency_ms: 803.13
The team decided to **use token‑based authentication** and to **test the authentication setup before deployment**.


In [18]:
print(ask_agent("who won the ballondor award of football?"))

TRACE | total_latency_ms: 885.77
The most recent Ballon d’Or winner (as of the 2023 ceremony) was **Lionel Messi**.


In [20]:
print(ask_agent("who won the ballondor award of football?"))

NameError: name 'MODEL' is not defined

In [22]:
print(ask_agent("who won the ballondor award of football?"))

TRACE | total_latency_ms: 413.53
I’m not sure which year you’re referring to. Could you let me know the specific year or season for the Ballon d’Or you’re interested in?


In [23]:
print(ask_agent("who won the ballondor award of football 2026?"))

TRACE | tool: search_meetings
TRACE | arguments: {'query': "Ballon d'Or 2026"}
TRACE | total_latency_ms: 721.75
I’m sorry, but I couldn’t find any records related to that.


In [25]:
print(ask_agent("who won the ballondor award of football?"))

TRACE | total_latency_ms: 619.38
I’m sorry, but I don’t have any records on that topic.
